In [3]:
import pyodbc
import json
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Load connection string
try:
    with open('jupytersettings.json', 'r') as file:
        settings = json.load(file)
        connection_string = settings['ConnectionStrings']['DefaultConnection']
except FileNotFoundError:
    print("Error: 'jupytersettings.json' file not found.")
    exit()
except KeyError:
    print("Error: 'ConnectionStrings' or 'DefaultConnection' key not found in JSON.")
    exit()
except Exception as e:
    print(f"Error reading 'jupytersettings.json': {e}")
    exit()

# Fetch available MarketTickers
market_tickers = []
try:
    conn = pyodbc.connect(connection_string)
    cursor = conn.cursor()
    query = "SELECT DISTINCT MarketTicker FROM [kalshibot-dev].[dbo].[t_Snapshots]"
    cursor.execute(query)
    market_tickers = [row[0] for row in cursor.fetchall()]
    cursor.close()
    conn.close()
except pyodbc.Error as e:
    print(f"Database error fetching MarketTickers: {e}")
    exit()
except Exception as e:
    print(f"Unexpected error fetching MarketTickers: {e}")
    exit()

if not market_tickers:
    print("No MarketTickers found in the database.")
    exit()

# Function to fetch data for a given MarketTicker
def fetch_data(market_ticker):
    try:
        conn = pyodbc.connect(connection_string)
        cursor = conn.cursor()
        query = "SELECT RawJson FROM [kalshibot-dev].[dbo].[t_Snapshots] WHERE MarketTicker = ?"
        cursor.execute(query, market_ticker)
        rows = cursor.fetchall()
        cursor.close()
        conn.close()
    except pyodbc.Error as e:
        print(f"Database error for MarketTicker '{market_ticker}': {e}")
        return None

    if not rows:
        print(f"No rows found for MarketTicker = {market_ticker}")
        return None

    # Deserialize JSON
    data = []
    skipped_rows = 0
    for row in rows:
        try:
            json_str = row[0]
            json_obj = json.loads(json_str)
            if "Markets" in json_obj and market_ticker in json_obj["Markets"]:
                market_data = json_obj["Markets"][market_ticker]
                market_data["Timestamp"] = json_obj.get("Timestamp")
                data.append(market_data)
            else:
                skipped_rows += 1
                print(f"Warning: 'Markets' or '{market_ticker}' missing in JSON for row")
        except json.JSONDecodeError as e:
            skipped_rows += 1
            print(f"Error deserializing JSON: {e}")
            continue

    print(f"Number of rows successfully parsed: {len(data)}")
    print(f"Number of rows skipped: {skipped_rows}")

    if not data:
        print(f"No valid JSON data found for MarketTicker = {market_ticker}")
        return None

    # Convert to DataFrame
    df = pd.DataFrame(data)
    if "Timestamp" in df.columns:
        df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    
    # Sample data if too large
    sample_size = 1000
    if len(df) > sample_size:
        print(f"Sampling {sample_size} rows for plotting (from {len(df)} total)")
        df = df.sample(n=sample_size, random_state=42)
    
    return df

# Create widgets
market_dropdown = widgets.Dropdown(
    options=market_tickers,
    description='Market:',
    value=market_tickers[0]
)

# Create 6 dropdowns for column selection
column_dropdowns = [
    widgets.Dropdown(
        options=['Loading...'],
        description=f'Column {i+1}:',
        disabled=True,
        layout={'width': '400px'}
    ) for i in range(6)
]

output = widgets.Output()

# Function to update column dropdowns based on selected market, preserving selections
def update_columns(market_ticker):
    df = fetch_data(market_ticker)
    if df is not None:
        # Get numeric columns (excluding Timestamp), sorted alphabetically
        numeric_cols = sorted(df.select_dtypes(include=['float64', 'int64']).columns)
        numeric_cols = [col for col in numeric_cols if col != 'Timestamp']
        
        # Store current selections
        current_selections = [dropdown.value for dropdown in column_dropdowns]
        
        # Update dropdown options
        for dropdown in column_dropdowns:
            dropdown.options = ['None'] + numeric_cols if numeric_cols else ['No numeric columns']
            dropdown.disabled = False if numeric_cols else True
        
        # Restore selections if possible, otherwise set defaults
        if numeric_cols:
            for i, dropdown in enumerate(column_dropdowns):
                if current_selections[i] in numeric_cols:
                    dropdown.value = current_selections[i]
                elif i == 0:
                    dropdown.value = 'BestYesAsk' if 'BestYesAsk' in numeric_cols else numeric_cols[0]
                elif i == 1:
                    dropdown.value = 'BestYesBid' if 'BestYesBid' in numeric_cols else (
                        numeric_cols[1] if len(numeric_cols) > 1 else 'None'
                    )
                else:
                    dropdown.value = 'None'
        else:
            for dropdown in column_dropdowns:
                dropdown.value = 'No numeric columns'
        return df
    else:
        for dropdown in column_dropdowns:
            dropdown.options = ['No data']
            dropdown.disabled = True
            dropdown.value = 'No data'
        return None

# Function to plot selected data with dual y-axes
def plot_data(market_ticker, columns):
    with output:
        clear_output(wait=True)
        df = fetch_data(market_ticker)
        if df is not None and 'Timestamp' in df.columns:
            columns = [col for col in columns if col in df.columns and col != 'None']
            if columns:
                fig, ax1 = plt.subplots(figsize=(12, 6))
                ax2 = ax1.twinx()  # Secondary y-axis
                
                # Define price columns (assumed to be in [0, 100])
                price_columns = ['BestYesAsk', 'BestYesBid']
                price_cols = [col for col in columns if col in price_columns]
                non_price_cols = [col for col in columns if col not in price_columns]
                
                # Plot price columns on y1 (left)
                y1_lines = []
                y1_labels = []
                if price_cols:
                    for col in price_cols:
                        line, = ax1.plot(df['Timestamp'], df[col], '-', label=col)
                        y1_lines.append(line)
                        y1_labels.append(col)
                    
                    # Calculate y1 limits with 10% margin, clamped to [0, 100]
                    y1_values = df[price_cols].values.flatten()
                    y1_min, y1_max = y1_values.min(), y1_values.max()
                    y1_range = y1_max - y1_min
                    margin = 0.1 * y1_range if y1_range > 0 else 0.1
                    y1_lower = max(0, y1_min - margin)
                    y1_upper = min(100, y1_max + margin)
                    ax1.set_ylim(y1_lower, y1_upper)
                    ax1.set_ylabel('Price')
                
                # Plot non-price columns on y2 (right)
                y2_lines = []
                y2_labels = []
                if non_price_cols:
                    for col in non_price_cols:
                        line, = ax2.plot(df['Timestamp'], df[col], '--', label=col)
                        y2_lines.append(line)
                        y2_labels.append(col)
                    
                    # Calculate y2 limits with 10% margin, dynamic scaling
                    y2_values = df[non_price_cols].values.flatten()
                    y2_min, y2_max = y2_values.min(), y2_values.max()
                    y2_range = y2_max - y2_min
                    margin = 0.1 * y2_range if y2_range > 0 else 0.1
                    y2_lower = y2_min - margin
                    y2_upper = y2_max + margin
                    ax2.set_ylim(y2_lower, y2_upper)
                    ax2.set_ylabel('Non-Price Values')
                
                # Create separate legends
                if y1_lines:
                    ax1.legend(y1_lines, y1_labels, loc='upper left', title='Price')
                if y2_lines:
                    ax2.legend(y2_lines, y2_labels, loc='upper right', title='Non-Price')
                
                # Finalize plot
                ax1.set_title(f'Data for {market_ticker}')
                ax1.set_xlabel('Timestamp')
                ax1.grid(True)
                plt.xticks(rotation=45)
                plt.tight_layout()
                plt.show()
            else:
                print(f"No valid columns selected for plotting in {market_ticker}")
        else:
            print(f"Cannot plot: Missing data or Timestamp for {market_ticker}")

# Handlers for widget changes
def on_market_change(change):
    df = update_columns(change['new'])
    if df is not None and column_dropdowns[0].options[0] != 'No numeric columns':
        plot_data(change['new'], [dropdown.value for dropdown in column_dropdowns])

def on_column_change(change):
    if any(dropdown.options[0] not in ['No numeric columns', 'No data'] for dropdown in column_dropdowns):
        plot_data(market_dropdown.value, [dropdown.value for dropdown in column_dropdowns])

# Connect handlers to widgets
market_dropdown.observe(on_market_change, names='value')
for dropdown in column_dropdowns:
    dropdown.observe(on_column_change, names='value')

# Display widgets
display(market_dropdown)
for dropdown in column_dropdowns:
    display(dropdown)
display(output)

# Initialize with first market
update_columns(market_dropdown.value)
if column_dropdowns[0].options[0] != 'No numeric columns':
    plot_data(market_dropdown.value, [dropdown.value for dropdown in column_dropdowns])

Dropdown(description='Market:', options=('KXPRESNOMD-28-GN', 'KXWAPOWHBAN-25', 'KXGREENIND-26', 'KXEPSTEIN-26'…

Dropdown(description='Column 1:', disabled=True, layout=Layout(width='400px'), options=('Loading...',), value=…

Dropdown(description='Column 2:', disabled=True, layout=Layout(width='400px'), options=('Loading...',), value=…

Dropdown(description='Column 3:', disabled=True, layout=Layout(width='400px'), options=('Loading...',), value=…

Dropdown(description='Column 4:', disabled=True, layout=Layout(width='400px'), options=('Loading...',), value=…

Dropdown(description='Column 5:', disabled=True, layout=Layout(width='400px'), options=('Loading...',), value=…

Dropdown(description='Column 6:', disabled=True, layout=Layout(width='400px'), options=('Loading...',), value=…

Output()

Number of rows successfully parsed: 120
Number of rows skipped: 0
